#Preliminary Analysis of Habitat Modification Scores 
####(main dataset is rss_tbl_river_habitat_survey - hms_class and subscores)

##River Habitat Survey HMS class

In [0]:
# Load packages
library(sparklyr) # connect to databricks
library(data.table) # data management
# GAMS - use mgcv for ordinal/catagorial data
#library(mgcv)
# Create a sparklyr connection.
sc <- spark_connect(method = "databricks")

##Load data
set parameters

In [0]:
# set parameters
param <- list(st="RSN")

# River habitat survey
# hms_class
rss_dt <- sdf_sql(sc, paste0("SELECT *
FROM prd_dash_lab.seda_restricted.rss_tbl_river_habitat_survey
WHERE network_id LIKE '", param$st, 
"%' AND completeness_band NOT LIKE '%Unacceptable%' ")) |> as.data.table()

In [0]:
# change to factors
rss_dt[, network_id := as.factor(network_id)]


In [0]:
#return only hms_class and hms_class_desc columns
rss_dt[, .(hms_class, hms_class_desc), keyby=hms_class] |> unique()

In [0]:
hms_class_table <- table(rss_dt[, hms_class])
# as %
hms_class_table  <- hms_class_table  / sum(hms_class_table) * 100
# Add names
classes <- c("Pristine/Near-natural", "Predominantly unmodified", "Obviously modified", "Significantly modified", "Severely modified")
names(hms_class_table) <- classes
# View table
hms_class_table

In [0]:
# plot overall percentages
par(bg="white", mar=c(5, 4, 4, 8))
cols <- c("#2ca25f", "#99d8c9", "#fec44f", "#fe9929", "#d95f0e")
# Plot
barplot(matrix(hms_class_table, ncol=1), col=cols,  ylim=c(0, 100), xlab="HMS Class")
# Legend
legend("right", inset=c(-0.45, 0), legend=names(hms_class_table),  fill=cols,  bty="n", xpd=TRUE)

In [0]:
# Alternative plot
par(bg="white")
barplot(matrix(hms_class_table, ncol=1), col=cols, beside=TRUE, ylim=c(0, 100), names.arg=names(hms_class_table), xlab="HMS Class", ylab="Proportion of Rivers (%)")

In [0]:
# Calculate overall % meeting high quality or above
hms_class_table[1] + hms_class_table[2]

###Calculate confidence intervals for the classes
*on the raw data

In [0]:
# Uses the binomial test to generate confidence intervals from the raw class data
# Thi gives class's proportion with a 95% CI
res <- rss_dt[, .N, by = hms_class][order(hms_class)]
res[, p := N / sum(N)]
res[, c("lo", "hi") := {
  ci <- sapply(N, \(n) binom.test(n, sum(N))$conf.int)
  .(ci[1, ], ci[2, ])
}]

# Remove row with na values for now.
res <- res[-6,]
res

In [0]:
# Alternative plot with confidence intervals
par(bg="white")
bar <- barplot(matrix(res[,p], ncol=1), col=cols, beside=TRUE, ylim=c(0, 1), names.arg=names(hms_class_table), xlab="HMS Class", ylab="Proportion of Rivers (%)")
# Add confidence intervals
arrows(x0=bar, y0=res[,lo], x1=bar, y1=res[,hi], code=3, angle=90, length=0.1)

##PCA - Identifying Habitat Subclasses Driving Modification Class

In [0]:
#list subclasses here as a vector
names(rss_dt)
hms_vars <- c("hms_reforced_bnk_bed_subscore", "hms_rsctned_bnk_subscore", "hms_berms_embnkments_subscore", "hms_culverts_subscore", "hms_wds_subscore", "hms_bridges_subscore", "hms_poaching_subscore", "hms_fords_subscore", "hms_ofall_dflector_subscore")

colSums(is.na(rss_dt[, ..hms_vars]))

#list the neworks that have missing data
rss_dt[is.na(hms_ofall_dflector_subscore),
       .(network_id, hms_class, survey_date)]

#remove NAs, create new data table
rss_pca <- rss_dt[complete.cases(rss_dt[, ..hms_vars])]

#PCA omitting the NAs
pca <- prcomp(rss_pca[, ..hms_vars], scale. = TRUE)


scores <- as.data.table(pca$x)
scores[, hms_class := rss_pca$hms_class]

summary(pca)

##Plot PCA

Scree plot - Variance explained by each principal component

In [0]:
#plot scree plot - shows how much variance each PC explains
plot(pca, type = "l")

In [0]:
Scatter plot - Principal component scores coloured by habitat modification class

In [0]:
scores <- as.data.table(pca$x)
scores[, hms_class := rss_pca$hms_class]

#plot PCA with arrows(subclasses)

scores[, hms_class_desc := factor(
  hms_class,
  levels = 1:5,
  labels = c(
   classes))]

cols <- c("darkgreen", "green", "yellow", "orange", "red")

#addiing arrows to pca plot
plot(scores$PC1, scores$PC2,
col = cols[scores$hms_class],pch = 16,xlab = "PC1 (19.0%)", ylab = "PC2 (16.0%)")

# Add arrows
arrows(0, 0, 
pca$rotation[,1] * 10,
pca$rotation[,2] * 8,
length = 0.1,col = "red")

# Add labels
text(
  pca$rotation[,1] * 10,
  pca$rotation[,2] * 8,
  labels = rownames(pca$rotation),
  col = "black",
  pos = 4,
  cex = 0.8)

legend(
  "bottomright",
  inset = c(-0.05, 0),
  legend = c(
    "Pristine/Near-natural",
    "Predominantly unmodified",
    "Obviously modified",
    "Significantly modified",
    "Severely modified"),
  col = cols, pch = 16, bty = "n", cex=0.8, xpd=TRUE)


Boxplot of PC1 by HMS Class - Distribution of PC1 scores across habitat modification classes

In [0]:
par(mar = c(10, 4, 4, 2))

boxplot(
  PC1 ~ hms_class_desc,
  data = scores,
  las = 2)

Contribution of HMS subclass scores to PCA axes

hms_reforced_bnk_bed_subscore, hms_wds_subscore and hms_bridges_subscore primarily associated with PC1 

hms_rsctned_bnk_subscore, hms_berms_embnkments_subscore and hms_poaching_subscore associated with PC2

In [0]:
pca$rotation

In [0]:
sort(abs(pca$rotation[,1]), decreasing = TRUE)


In [0]:
sort(abs(pca$rotation[,2]), decreasing = TRUE)

Correlation test- using both Spearman and Kendall methods (just as a test)

In [0]:
cor.test(
  as.numeric(scores$hms_class),
  scores$PC1,
  method = "spearman"
)

In [0]:
cor.test(
  as.numeric(scores$hms_class),
  scores$PC1,
  method = "kendall")

In [0]:
names(scores)

Both Spearman and Kendall show a strong positive relationship association is strong (tau and rho) and highly significant (p < 0.001)